# A_4_3 — Zero-shot classification

**Input:** `Results/Delay_reason_only*.xlsx`, expert taxonomy
**Output:** classified delay and rescheduling statements

Classifies every statement into one category of the taxonomy using a locally
hosted Llama 3.2 model, with deterministic decoding. Three rules are given in
the prompt: assign by root cause rather than downstream consequence, prefer the
stated rationale over boilerplate, and rely on the underlying driver rather than
the promoter's wording. Outputs without a valid label are flagged rather than
forced into a category.

These labels are not the ones reported in the paper: every assignment was
reviewed manually, and A_5 reads the reviewed files.

In [ ]:
import pandas as pd
import requests
import subprocess
import umap
import hdbscan
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from tqdm import tqdm 

In [15]:
df_delayed = pd.read_excel("Results/Delay_reason_onlyDelayed.xlsx")

df_delayed = df_delayed.dropna(subset=['Inv_Progress Driver'])
df_delayed['Inv_Progress Driver'] = df_delayed['Inv_Progress Driver'].astype(str)

sentences_delayed = df_delayed['Inv_Progress Driver'].tolist()
print(f"Successfully loaded {len(sentences_delayed)} sentences for the 'Delayed' analysis.")

df_rescheduled = pd.read_excel("Results/Delay_reason_onlyRescheduled.xlsx")

df_rescheduled = df_rescheduled.dropna(subset=['Inv_Progress Driver'])
df_rescheduled['Inv_Progress Driver'] = df_rescheduled['Inv_Progress Driver'].astype(str)

sentences_rescheduled = df_rescheduled['Inv_Progress Driver'].tolist()
print(f"Successfully loaded {len(sentences_rescheduled)} sentences for the 'Rescheduled' analysis.")

Successfully loaded 479 sentences for the 'Delayed' analysis.
Successfully loaded 213 sentences for the 'Rescheduled' analysis.


In [16]:
print("Loading the SentenceTransformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("\nGenerating embeddings for Delayed sentences...")
embs_delayed = embedder.encode(sentences_delayed, show_progress_bar=False)

print("Generating embeddings for Rescheduled sentences...")
embs_rescheduled = embedder.encode(sentences_rescheduled, show_progress_bar=False)

print("\nSuccess! Both datasets have been successfully embedded.")

Loading the SentenceTransformer model...

Generating embeddings for Delayed sentences...
Generating embeddings for Rescheduled sentences...

Success! Both datasets have been successfully embedded.


In [17]:
taxonomy_delayed = {
    "ENV_PERMIT_EIA": "Delays specifically triggered by Environmental Impact Assessments (EIA), ecological constraints, or the subsequent need for spatial re-routing to mitigate environmental impact.",
    "SOCIAL_AND_ROUTING": "Delays initiated by local social opposition, public resistance, and community-driven re-routing demands.",
    "LAND_ACQUISITION": "Delays confined to the physical and legal expropriation of property or securing right-of-way. Excludes broad land-use planning modifications.",
    "LEGAL_AND_REGULATORY": "Delays strictly caused by top-down legislative changes, unclear national regulatory frameworks, court rulings, or missing national development plan approvals.",
    "GENERAL_BUREAUCRACY": "Delays stemming from general administrative friction, multi-stage local permitting, and land-use plan adaptations (e.g., GRUP). Acts as the classification for permitting delays not explicitly categorized as environmental or legal.",
    "FUNDING_AND_FINANCE": "Delays related solely to capital procurement, budget deficits, or financial closure. Explicitly excludes supply chain contracts or inter-project dependency agreements.",
    "CASCADING_DELAYS": "Delays forced by exogenous dependencies, such as waiting for correlated grid investments, third-party renewable energy contracts, or the commissioning of new power plants.",
    "PROCUREMENT_AND_SUPPLY_CHAIN": "Delays rooted in the industrial supply chain, including tendering failures, contractor insolvency, and material shortages.",
    "SCOPE_AND_DESIGN_CHANGES": "Delays driven by technical engineering adjustments, feasibility study outcomes, or revisions in planning input data.",
    "NO_DELAY_OR_DATA_ISSUE": "Entries indicating the project progressed as planned, or containing invalid, empty, or uninterpretable data."
}

taxonomy_rescheduled = {
    "SYSTEM_DEPENDENCIES": "Rescheduling driven by operational dependencies on other grid elements, alignment with neighboring lines, shifts in macro demand scenarios, changes in upstream generation background (RES, nuclear, hydro, offshore), or grid connection timelines.",
    "REGULATORY_PLANNING": "Top-down rescheduling caused strictly by regulatory approvals, CBCA decisions, master plan updates, intergovernmental agreements, or changes in national/European energy directives.",
    "TECHNICAL_SCOPE_AND_STUDIES": "Projects rescheduled pending feasibility studies, changes in technical scope/design, testing novel solutions, or contracting/EPC crises (e.g., COVID).",
    "SITING_AND_ACQUISITION": "Difficulties and delays strictly related to acquiring land, securing right-of-way, or physical routing constraints.",
    "NO_CHANGE_OR_DATA_ISSUE": "Use ONLY for completely empty data, hyphens, or brief statements indicating absolutely no change."
}


In [19]:
def classify_delayed_with_llm(sentence: str, taxonomy: dict, model: str = "llama3.2") -> str:
    prompt = (
        "You are an expert Data Annotator for a high-impact academic paper on energy infrastructure.\n"
        "Your task is to classify the provided text into EXACTLY ONE of the official categories.\n\n"
        "OFFICIAL TAXONOMY:\n"
    )
    for cat_name, cat_desc in taxonomy.items():
        prompt += f"- {cat_name}: {cat_desc}\n"
        
    prompt += (
        "\nCRITICAL ANALYTICAL RULES:\n"
        "1. ROOT CAUSE WINS: In presence of compound sentences describing a chain of events, classify strictly based on the primary chronological trigger (Root Cause), ignoring downstream consequences.\n"
        "2. STRICT FORMATTING: Output ONLY the exact category name. Do not add brackets, punctuation, or explanations.\n\n"
        "EXAMPLES:\n"
        "Text: \"Realization of this project is delayed because of problems with permits. Now we are closing financial structure.\"\n"
        "Output: GENERAL_BUREAUCRACY\n\n"
        "Text: \"Awaiting international contracts for renewable energy\"\n"
        "Output: CASCADING_DELAYS\n\n"
        f"TEXT TO CLASSIFY:\n\"{sentence}\"\n\n"
        "Output:"
    )

    return _call_ollama_api(prompt, taxonomy, model)


def classify_rescheduled_with_llm(sentence: str, taxonomy: dict, model: str = "llama3.2") -> str:
    prompt = (
        "You are an expert Data Annotator for a high-impact academic paper on energy infrastructure.\n"
        "Your task is to classify the provided text into EXACTLY ONE of the official categories.\n\n"
        "OFFICIAL TAXONOMY:\n"
    )
    for cat_name, cat_desc in taxonomy.items():
        prompt += f"- {cat_name}: {cat_desc}\n"
        
    prompt += (
        "\nCRITICAL ANALYTICAL RULES:\n"
        "1. RATIONALE SUPREMACY: Focus on the underlying driver of the modification. If a text contains contradictory boilerplate (e.g., 'progressed as planned') but subsequently provides a rationale for rescheduling, the rationale strictly dictates the classification.\n"
        "2. INTENT OVER LEXICON: Evaluate the core driver even if the promoter uses informal wording like 'delay' or 'postponed' within a rescheduled dataset context.\n"
        "3. STRICT FORMATTING: Output ONLY the exact category name. Do not add brackets, punctuation, or explanations.\n\n"
        "EXAMPLES:\n"
        "Text: \"Investment progresses as planned, rescheduled slightly due to expected development on the drivers.\"\n"
        "Output: SYSTEM_DEPENDENCIES\n\n"
        "Text: \"Delay due to missing confirmation by the regulator\"\n"
        "Output: REGULATORY_PLANNING\n\n"
        f"TEXT TO CLASSIFY:\n\"{sentence}\"\n\n"
        "Output:"
    )

    return _call_ollama_api(prompt, taxonomy, model)

def _call_ollama_api(prompt: str, taxonomy: dict, model: str) -> str:
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature":0.0, "seed":42}}
        )
        response.raise_for_status()
        result = response.json().get("response", "").strip()
        
        result = result.replace("[", "").replace("]", "").replace(".", "").replace(":", "").strip()
        
        for valid_key in taxonomy.keys():
            if valid_key in result:
                return valid_key
                
        return "UNCLASSIFIED"
    except Exception as e:
        return "API_ERROR"

print("\n--- Starting LLM Classification for DELAYED Projects ---")
delayed_results = []

for text in tqdm(sentences_delayed, desc="Classifying Delayed"):
    if pd.isna(text) or text.strip() == "":
        delayed_results.append("NO_DELAY_OR_DATA_ISSUE")
    else:
        cat = classify_delayed_with_llm(text, taxonomy_delayed)
        delayed_results.append(cat)

df_delayed['Expert_Category'] = delayed_results
df_delayed.to_excel("Results/Delayed_Final_llama3.2.xlsx", index=False)
print("Saved Delayed classification results to Excel.")

print("\n--- Starting LLM Classification for RESCHEDULED Projects ---")
rescheduled_results = []

for text in tqdm(sentences_rescheduled, desc="Classifying Rescheduled"):
    if pd.isna(text) or text.strip() == "":
        rescheduled_results.append("NO_CHANGE_OR_DATA_ISSUE")
    else:
        cat = classify_rescheduled_with_llm(text, taxonomy_rescheduled)
        rescheduled_results.append(cat)

df_rescheduled['Expert_Category'] = rescheduled_results
df_rescheduled.to_excel("Results/Rescheduled_Final_llama3.2.xlsx", index=False)
print("Saved Rescheduled classification results to Excel.")


--- Starting LLM Classification for DELAYED Projects ---


Classifying Delayed: 100%|██████████| 479/479 [00:42<00:00, 11.23it/s]


Saved Delayed classification results to Excel.

--- Starting LLM Classification for RESCHEDULED Projects ---


Classifying Rescheduled: 100%|██████████| 213/213 [00:18<00:00, 11.26it/s]

Saved Rescheduled classification results to Excel.
